# Orchestador Principal - Modelos de Recomendación Anime

Este notebook se encarga de unificar el proceso de carga de datos, instanciación y comparación de todas las arquitecturas de recomendación (KNN, PMF, BMF, NCF).

In [6]:
import polars as pl
import pandas as pd
import numpy as np
import pickle
import scipy.sparse as sp
from collections import defaultdict

import warnings
warnings.filterwarnings('ignore')

## 1. Carga de Datos y Mapeos

In [7]:
# Cargar DataFrames en Pandas (KNN usa set y diccionarios de python que iteran mas comodo sobre pd o tuples)
df_train_pl = pl.read_parquet("data/train.parquet")
df_test_pl = pl.read_parquet("data/test.parquet")

df_train = df_train_pl.to_pandas()
df_test = df_test_pl.to_pandas()

# Cargar los mapeos guardados por preprocess.py
with open("data/mapeos.pkl", "rb") as f:
    mapeos = pickle.load(f)

user2idx = mapeos['user2idx']
anime2idx = mapeos['anime2idx']

NUM_USERS = len(user2idx)
NUM_ITEMS = len(anime2idx)

print(f"Usuarios únicos: {NUM_USERS}")
print(f"Items únicos: {NUM_ITEMS}")
print(f"Interacciones Train: {len(df_train):,}")
print(f"Interacciones Test: {len(df_test):,}")

Usuarios únicos: 47143
Items únicos: 6532
Interacciones Train: 4,915,940
Interacciones Test: 1,228,986


## 2. Preparación para PMF (Matrices Dispersas)

In [8]:
# PMF necesita las matrices dispersas CSR generadas a partir de ratings (1 a 10)
R_train_sparse = sp.csr_matrix(
    (df_train['rating'].values, (df_train['user_id'].values, df_train['anime_id'].values)), 
    shape=(NUM_USERS, NUM_ITEMS)
)

R_test_sparse = sp.csr_matrix(
    (df_test['rating'].values, (df_test['user_id'].values, df_test['anime_id'].values)), 
    shape=(NUM_USERS, NUM_ITEMS)
)

MU = df_train['rating'].mean()
print(f"Media global de entrenamiento (MU): {MU:.4f}")

Media global de entrenamiento (MU): 7.7929


## 3. Modelo 1 - KNN (K-Nearest Neighbors)

In [ ]:
from knn import run_knn

df_results_knn, knn_model = run_knn(df_train, df_test.sample(1500, random_state=42), k_values=[5, 10, 20, 30, 50, 75, 100])
display(df_results_knn)


Cargando resultados de K guardados previamente desde results/resultados_k_optimo.csv...
     K      RMSE       MAE  Cobertura
0    5  1.339785  1.016814      100.0
1   10  1.339077  1.001653      100.0
2   20  1.358241  1.015117      100.0
3   30  1.370351  1.020646      100.0
4   50  1.396968  1.038031      100.0
5   75  1.422264  1.056399      100.0
6  100  1.439578  1.070578      100.0
>> Inicializando modelo KNN (sin revaluar)...
Construyendo matriz interna KNN...


,K,RMSE,MAE,Cobertura
0,5,1.339785,1.016814,100.0
1,10,1.339077,1.001653,100.0
2,20,1.358241,1.015117,100.0
3,30,1.370351,1.020646,100.0
4,50,1.396968,1.038031,100.0
5,75,1.422264,1.056399,100.0
6,100,1.439578,1.070578,100.0


## 4. Modelo 2 - PMF (Probabilistic Matrix Factorization)

In [ ]:
from pmf import run_pmf

pmf_history, pmf_best_rmse, pmf_model = run_pmf(
    R_train_sparse, 
    R_test_sparse, 
    mu=MU, 
    n_users=NUM_USERS, 
    n_items=NUM_ITEMS, 
    n_factors=50, 
    epochs=30, 
    patience=5
)

print(f"El mejor RMSE logrado por PMF fue: {pmf_best_rmse:.4f}")

>> Inicializando modelo PMF...


KeyboardInterrupt: 